In [ ]:
# Cell 1 — Install
!pip install ultralytics -q

In [ ]:
# Cell 2 — Mount Drive + giải nén
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os, glob

# Tìm file zip trên Drive nếu chưa biết tên
zips = glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
print('Zips found:', zips)

zip_path   = '/content/drive/MyDrive/RR2.yolov8.zip'  # ← đổi nếu tên khác
extract_to = '/content/dataset'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)

yamls = glob.glob(extract_to + '/**/data.yaml', recursive=True)
print('YAML found:', yamls)

In [ ]:
# Cell 3 — Tạo valid + sửa data.yaml
import os, shutil, glob, yaml

yamls = glob.glob('/content/dataset/**/data.yaml', recursive=True)
dataset_root = os.path.dirname(yamls[0])
yaml_path    = yamls[0]
print('Root:', dataset_root)

# Tạo valid từ 10 ảnh train nếu chưa có
os.makedirs(dataset_root + '/valid/images', exist_ok=True)
os.makedirs(dataset_root + '/valid/labels', exist_ok=True)

if len(os.listdir(dataset_root + '/valid/images')) == 0:
    imgs = sorted(os.listdir(dataset_root + '/train/images'))[:10]
    for f in imgs:
        shutil.copy(dataset_root + '/train/images/' + f,
                    dataset_root + '/valid/images/' + f)
        lbl = f.rsplit('.', 1)[0] + '.txt'
        src = dataset_root + '/train/labels/' + lbl
        if os.path.exists(src):
            shutil.copy(src, dataset_root + '/valid/labels/' + lbl)
    print(f'Created valid with {len(imgs)} images')

# Cập nhật path tuyệt đối trong data.yaml
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['train'] = dataset_root + '/train/images'
cfg['val']   = dataset_root + '/valid/images'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print('train:', len(os.listdir(dataset_root + '/train/images')), 'images')
print('valid:', len(os.listdir(dataset_root + '/valid/images')), 'images')
print('nc:   ', cfg['nc'])
print('names:', cfg['names'])

In [ ]:
# Cell 4 — Train (đảm bảo Runtime = GPU T4)
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data=yaml_path, epochs=100, imgsz=512, device=0)

In [ ]:
# Cell 5 — Export ONNX + download về máy
from ultralytics import YOLO
from google.colab import files
import glob

pts = glob.glob('/content/runs/detect/train*/weights/best.pt')
print('Found:', pts)
pt = sorted(pts)[-1]

model = YOLO(pt)
model.export(format='onnx', imgsz=512)

onnx = pt.replace('best.pt', 'best.onnx')
print('Downloading:', onnx)
files.download(onnx)